### Célula 1 — Registrando as tabelas Gold no Catalog

In [0]:
import pyspark.sql.functions as F

GOLD_PATH = "/Volumes/workspace/default/raw/gold/"                  # caminho base da Gold

# Registrando cada tabela como view temporária para consulta SQL
tabelas = [
    "f_customer_support_tickets",
    "dim_customer",
    "dim_product",
    "dim_type",
    "dim_subject",
    "dim_status",
    "dim_priority",
    "dim_channel",
    "dim_ticket_description",
    "dim_calendario"
]

for tabela in tabelas:
    df = spark.read.format("delta").load(f"{GOLD_PATH}{tabela}/")   # lê o Delta Lake
    df.createOrReplaceTempView(tabela)                               # registra como view SQL
    print(f"✅ {tabela} registrada!")

print()
print("🎉 Todas as tabelas disponíveis para SQL!")

✅ f_customer_support_tickets registrada!
✅ dim_customer registrada!
✅ dim_product registrada!
✅ dim_type registrada!
✅ dim_subject registrada!
✅ dim_status registrada!
✅ dim_priority registrada!
✅ dim_channel registrada!
✅ dim_ticket_description registrada!
✅ dim_calendario registrada!

🎉 Todas as tabelas disponíveis para SQL!


### Célula 2 — Visão geral da tabela fato

In [0]:
%sql
-- Visão geral da tabela fato
SELECT
    COUNT(*)                                    AS total_tickets,
    COUNT(DISTINCT Customer_ID)                 AS clientes_unicos,
    COUNT(DISTINCT Product_ID)                  AS produtos_unicos,
    ROUND(AVG(Customer_Satisfaction_Rating), 2) AS satisfacao_media,
    ROUND(SUM(Is_Resolved) / COUNT(*) * 100, 1) AS taxa_resolucao_pct,
    MIN(Date_of_Purchase)                       AS data_inicio,
    MAX(Date_of_Purchase)                       AS data_fim
FROM f_customer_support_tickets

total_tickets,clientes_unicos,produtos_unicos,satisfacao_media,taxa_resolucao_pct,data_inicio,data_fim
8469,8320,42,2.99,32.7,2020-01-01,2021-12-30


### Célula 3 — Satisfação média por canal e tipo

In [0]:
%sql
-- Satisfação média cruzada: canal x tipo de ticket
SELECT
    dc.Ticket_Channel                               AS canal,
    dt.Ticket_Type                                  AS tipo,
    COUNT(*)                                        AS total_tickets,
    ROUND(AVG(f.Customer_Satisfaction_Rating), 2)   AS satisfacao_media,
    ROUND(SUM(f.Is_Resolved) / COUNT(*) * 100, 1)  AS taxa_resolucao_pct
FROM f_customer_support_tickets f
JOIN dim_channel  dc ON f.Channel_ID  = dc.Channel_ID
JOIN dim_type     dt ON f.Type_ID     = dt.Type_ID
WHERE f.Customer_Satisfaction_Rating IS NOT NULL
GROUP BY dc.Ticket_Channel, dt.Ticket_Type
ORDER BY satisfacao_media ASC

canal,tipo,total_tickets,satisfacao_media,taxa_resolucao_pct
Social media,Technical issue,152,2.76,100.0
Phone,Refund request,138,2.8,100.0
Email,Cancellation request,154,2.86,100.0
Phone,Billing inquiry,137,2.92,100.0
Phone,Technical issue,151,2.94,100.0
Social media,Refund request,144,2.94,100.0
Email,Product inquiry,126,2.94,100.0
Social media,Product inquiry,144,2.96,100.0
Email,Refund request,171,2.99,100.0
Chat,Refund request,143,2.99,100.0


#### Query 2 — Satisfação por Canal x Tipo

#### Top 3 Piores Combinações 🔴
| Canal | Tipo | Satisfação |
|-------|------|-----------|
| Social Media | Technical Issue | 2.76 |
| Phone | Refund Request | 2.80 |
| Email | Cancellation Request | 2.86 |

#### Top 3 Melhores Combinações ✅
| Canal | Tipo | Satisfação |
|-------|------|-----------|
| Social Media | Cancellation Request | 3.24 |
| Chat | Technical Issue | 3.15 |
| Chat | Billing Inquiry | 3.13 |

#### Insights
- **Social Media + Technical Issue = 2.76** — pior combinação do negócio
- **Chat performa bem em todos os tipos** — 4 das 5 melhores combinações são Chat
- **Phone + Refund Request = 2.80** — cliente frustrado + canal que gera espera
- Taxa de resolução 100% em todos — só tickets resolvidos têm avaliação

### Célula 4 — Ranking de produtos por insatisfação

In [0]:
%sql
-- Top 10 produtos com menor satisfação e maior volume
SELECT
    dp.Product_Purchased                            AS produto,
    COUNT(*)                                        AS total_tickets,
    ROUND(AVG(f.Customer_Satisfaction_Rating), 2)   AS satisfacao_media,
    ROUND(SUM(f.Is_Resolved) / COUNT(*) * 100, 1)  AS taxa_resolucao_pct,
    ROUND(AVG(f.Resolution_Time_Hours), 1)          AS tempo_medio_horas
FROM f_customer_support_tickets f
JOIN dim_product dp ON f.Product_ID = dp.Product_ID
WHERE f.Customer_Satisfaction_Rating IS NOT NULL
GROUP BY dp.Product_Purchased
ORDER BY satisfacao_media ASC
LIMIT 10

produto,total_tickets,satisfacao_media,taxa_resolucao_pct,tempo_medio_horas
Fitbit Versa Smartwatch,63,2.54,100.0,7.9
Sony Xperia,78,2.69,100.0,7.6
Dell XPS,48,2.75,100.0,8.2
GoPro Action Camera,59,2.75,100.0,7.0
LG OLED,73,2.77,100.0,8.1
Microsoft Xbox Controller,58,2.79,100.0,7.9
Apple AirPods,74,2.81,100.0,8.0
Bose QuietComfort,68,2.82,100.0,7.4
Nintendo Switch,56,2.86,100.0,8.9
PlayStation,65,2.88,100.0,8.5


#### Query 3 — Top 10 Produtos com Menor Satisfação

| Produto | Tickets | Satisfação | Tempo Resolução |
|---------|---------|-----------|-----------------|
| Fitbit Versa Smartwatch | 63 | 2.54 🔴 | 7.9h |
| Sony Xperia | 78 | 2.69 🔴 | 7.6h |
| Dell XPS | 48 | 2.75 🔴 | 8.2h |
| GoPro Action Camera | 59 | 2.75 🔴 | 7.0h |
| LG OLED | 73 | 2.77 🔴 | 8.1h |
| Microsoft Xbox Controller | 58 | 2.79 🔴 | 7.9h |
| Apple AirPods | 74 | 2.81 🔴 | 8.0h |
| Bose QuietComfort | 68 | 2.82 🔴 | 7.4h |
| Nintendo Switch | 56 | 2.86 🔴 | 8.9h |
| PlayStation | 65 | 2.88 🔴 | 8.5h |

#### Insights
- **Fitbit Versa lidera a insatisfação (2.54)** — muito abaixo da média geral 2.99
- **Nintendo Switch tem maior tempo de resolução (8.9h)** — complexidade técnica alta
- **Todos abaixo de 3.0** — nenhum produto nesse ranking atinge a meta
- **Sony Xperia com 78 tickets e 2.69** — alto volume + baixa satisfação = prioridade máxima
- Produtos de **áudio e gaming** dominam o ranking — suporte técnico especializado necessário

### Célula 5 — Evolução mensal da satisfação

In [0]:
%sql
-- Evolução mensal da satisfação média
SELECT
    dc.Ano                                          AS ano,
    dc.Mes_nome                                     AS mes,
    f.Purchase_Month                                AS mes_num,
    COUNT(*)                                        AS total_avaliados,
    ROUND(AVG(f.Customer_Satisfaction_Rating), 2)   AS satisfacao_media,
    ROUND(SUM(f.Is_Resolved) / COUNT(*) * 100, 1)  AS taxa_resolucao_pct
FROM f_customer_support_tickets f
JOIN dim_calendario dc ON f.Date_of_Purchase = dc.Date
WHERE f.Customer_Satisfaction_Rating IS NOT NULL
GROUP BY dc.Ano, dc.Mes_nome, f.Purchase_Month
ORDER BY dc.Ano, f.Purchase_Month

ano,mes,mes_num,total_avaliados,satisfacao_media,taxa_resolucao_pct
2020,January,1,127,2.89,100.0
2020,February,2,123,3.11,100.0
2020,March,3,105,3.22,100.0
2020,April,4,123,3.03,100.0
2020,May,5,96,3.08,100.0
2020,June,6,124,3.05,100.0
2020,July,7,119,3.29,100.0
2020,August,8,96,3.0,100.0
2020,September,9,139,3.12,100.0
2020,October,10,120,2.98,100.0


In [0]:
%sql
-- Evolução mensal SEM filtro de satisfação
SELECT
    dc.Ano                                          AS ano,
    dc.Mes_nome                                     AS mes,
    f.Purchase_Month                                AS mes_num,
    COUNT(*)                                        AS total_tickets,
    SUM(f.Is_Resolved)                              AS resolvidos,
    ROUND(AVG(f.Customer_Satisfaction_Rating), 2)   AS satisfacao_media,
    ROUND(SUM(f.Is_Resolved) / COUNT(*) * 100, 1)  AS taxa_resolucao_pct
FROM f_customer_support_tickets f
JOIN dim_calendario dc ON f.Date_of_Purchase = dc.Date
GROUP BY dc.Ano, dc.Mes_nome, f.Purchase_Month
ORDER BY dc.Ano, f.Purchase_Month

ano,mes,mes_num,total_tickets,resolvidos,satisfacao_media,taxa_resolucao_pct
2020,January,1,377,127,2.89,33.7
2020,February,2,376,123,3.11,32.7
2020,March,3,324,105,3.22,32.4
2020,April,4,354,123,3.03,34.7
2020,May,5,322,96,3.08,29.8
2020,June,6,358,124,3.05,34.6
2020,July,7,366,119,3.29,32.5
2020,August,8,327,96,3.0,29.4
2020,September,9,369,139,3.12,37.7
2020,October,10,373,120,2.98,32.2


#### Query 4 — Evolução Mensal da Satisfação

#### 2020 vs 2021 — Comparativo
| Período | Satisfação Média |
|---------|-----------------|
| 2020 | 3.04 ✅ |
| 2021 | 2.94 🔴 |

#### Piores meses 🔴
| Mês | Satisfação |
|-----|-----------|
| Fev/2021 | 2.68 |
| Nov/2020 | 2.70 |
| Set/2021 | 2.74 |

#### Melhores meses ✅
| Mês | Satisfação |
|-----|-----------|
| Jul/2020 | 3.29 |
| Mar/2020 | 3.22 |
| Abr/2021 | 3.19 |

#### Insights
- **2020 média 3.04 vs 2021 média 2.94** — deterioração ao longo do tempo 🔴
- **2021 teve queda consistente de Jul a Out** — 4 meses consecutivos abaixo de 2.90
- **Dez/2021 (3.15) encerra bem** — possível recuperação no início de 2022
- Tendência de **piora ao longo do tempo** — sem intervenção vai continuar caindo

### Célula 6 — Análise de clientes por faixa etária

In [0]:
%sql
-- Satisfação e volume por faixa etária
SELECT
    dc.Age_Group                                    AS faixa_etaria,
    COUNT(*)                                        AS total_tickets,
    COUNT(DISTINCT f.Customer_ID)                   AS clientes_unicos,
    ROUND(AVG(f.Customer_Satisfaction_Rating), 2)   AS satisfacao_media,
    ROUND(SUM(f.Is_Resolved) / COUNT(*) * 100, 1)  AS taxa_resolucao_pct,
    ROUND(AVG(f.Resolution_Time_Hours), 1)          AS tempo_medio_horas
FROM f_customer_support_tickets f
JOIN dim_customer dc ON f.Customer_ID = dc.Customer_ID
WHERE f.Customer_Satisfaction_Rating IS NOT NULL
GROUP BY dc.Age_Group
ORDER BY satisfacao_media ASC

faixa_etaria,total_tickets,clientes_unicos,satisfacao_media,taxa_resolucao_pct,tempo_medio_horas
51+,1068,1063,2.96,100.0,7.7
18-25,382,380,2.99,100.0,7.4
36-50,800,790,3.0,100.0,7.9
26-35,519,516,3.04,100.0,7.7
